# 02 — Preprocessing & Feature Engineering

This notebook:
- loads the raw CSV
- cleans + engineers features (month cyclical, age bins, domestic/youth flags, etc.)
- creates a stratified train/val/test split
- saves splits to `data/processed/` as parquet

Target: `Crime Solved` → {0,1}

In [1]:
# Setup
from pathlib import Path
import sys

import pandas as pd

# Resolve project root (works when running from workspace root or /notebooks)
_cwd = Path.cwd()
PROJECT_ROOT = _cwd if (_cwd / "data").exists() else _cwd.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    DataPaths,
    RAW_TARGET_COL,
    load_raw_csv,
    make_target,
    prepare_dataframe_for_modeling,
    save_splits,
    stratified_split,
)

paths = DataPaths.from_project_root(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
print("Raw CSV:", paths.raw_csv)
print("Processed dir:", paths.processed_dir)

Project root: c:\Users\Wassim Rahali\Desktop\Crime IA
Raw CSV: c:\Users\Wassim Rahali\Desktop\Crime IA\data\raw\database.csv
Processed dir: c:\Users\Wassim Rahali\Desktop\Crime IA\data\processed


In [2]:
# Load raw + engineer features (no encoding yet)
df_raw = load_raw_csv(paths.raw_csv, low_memory=False)
print("Raw shape:", df_raw.shape)

# This adds engineered columns and drops IDs / leakage-ish fields
df_feat = prepare_dataframe_for_modeling(df_raw)
print("Feature DF shape:", df_feat.shape)

# Build target and filter out unknown targets
y = make_target(df_feat, target_col=RAW_TARGET_COL)
mask = y.notna()
X = df_feat.loc[mask].drop(columns=[RAW_TARGET_COL]).reset_index(drop=True)
y = y.loc[mask].astype(int).reset_index(drop=True)

print("Modeling X shape:", X.shape)
print("Target mean:", float(y.mean()))
X.head()

Raw shape: (638454, 24)
Feature DF shape: (638454, 32)
Modeling X shape: (638454, 31)
Target mean: 0.7019644328330623


,Agency Code,Agency Type,City,State,Year,Month,Incident,Crime Type,Victim Sex,Victim Age,...,month_cos,Season,VictimAgeBin,PerpetratorAgeBin,age_difference,is_youth_victim,same_race,same_sex,is_domestic,multi_victim
0,AK00101,Municipal Police,Anchorage,Alaska,1980,NaN,1,Murder or Manslaughter,Male,14,...,NaN,<NA>,13-17,13-17,1.0,1,1,1,0,0
1,AK00101,Municipal Police,Anchorage,Alaska,1980,NaN,1,Murder or Manslaughter,Male,43,...,NaN,<NA>,41-60,41-60,-1.0,0,1,1,0,0
2,AK00101,Municipal Police,Anchorage,Alaska,1980,NaN,2,Murder or Manslaughter,Female,30,...,NaN,<NA>,26-40,NaN,NaN,0,0,0,0,0
3,AK00101,Municipal Police,Anchorage,Alaska,1980,NaN,1,Murder or Manslaughter,Male,43,...,NaN,<NA>,41-60,41-60,-1.0,0,1,1,0,0
4,AK00101,Municipal Police,Anchorage,Alaska,1980,NaN,2,Murder or Manslaughter,Female,30,...,NaN,<NA>,26-40,NaN,NaN,0,0,0,0,0


In [3]:
# Stratified split + persistence
assert "paths" in globals(), "Run the Setup cell (Cell 2) first so imports/paths are configured."
assert "stratified_split" in globals(), "Run the Setup cell (Cell 2) first."
assert "save_splits" in globals(), "Run the Setup cell (Cell 2) first."

# Stratified split (70/15/15)
X_train, X_val, X_test, y_train, y_val, y_test = stratified_split(
    X, y, test_size=0.30, val_size=0.50, random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

# Persist engineered splits (with target column included)
save_splits(
    paths.processed_dir,
    X_train=X_train,
    X_val=X_val,
    X_test=X_test,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
    format="parquet",
)

print("Saved to:", paths.processed_dir)

Train: (446917, 31) (446917,)
Val: (95768, 31) (95768,)
Test: (95769, 31) (95769,)
Saved to: c:\Users\Wassim Rahali\Desktop\Crime IA\data\processed


In [4]:
# Verify saved splits load correctly
from src.preprocessing import load_processed_split

train_df = load_processed_split(paths.processed_dir, "train")
val_df = load_processed_split(paths.processed_dir, "val")
test_df = load_processed_split(paths.processed_dir, "test")

print("Train file shape:", train_df.shape)
print("Val file shape:", val_df.shape)
print("Test file shape:", test_df.shape)

assert RAW_TARGET_COL in train_df.columns
train_df.head()

Train file shape: (446917, 32)
Val file shape: (95768, 32)
Test file shape: (95769, 32)


,Agency Code,Agency Type,City,State,Year,Month,Incident,Crime Type,Victim Sex,Victim Age,...,Season,VictimAgeBin,PerpetratorAgeBin,age_difference,is_youth_victim,same_race,same_sex,is_domestic,multi_victim,Crime Solved
0,VA03500,Sheriff,Giles,Virginia,1982,NaN,1,Murder or Manslaughter,Female,45,...,None,41-60,41-60,-3.0,0,1,0,1,0,1
1,TNMPD00,Municipal Police,Shelby,Tennessee,2001,NaN,10,Murder or Manslaughter,Male,46,...,None,41-60,26-40,-6.0,0,1,1,0,0,1
2,CA01942,Municipal Police,Los Angeles,California,1984,NaN,7,Murder or Manslaughter,Male,22,...,None,18-25,NaN,NaN,0,0,0,0,0,0
3,NY03030,Municipal Police,New York,New York,1997,NaN,24,Murder or Manslaughter,Male,20,...,None,18-25,NaN,NaN,0,0,0,0,0,0
4,ME009SP,State Police,Oxford,Maine,2005,NaN,2,Murder or Manslaughter,Male,29,...,None,26-40,41-60,27.0,0,1,1,1,0,1


In [5]:
# Preview preprocessing feature space size (encoding)
from src.preprocessing import build_preprocessor, get_feature_names

preprocessor = build_preprocessor(X_train)
preprocessor.fit(X_train, y_train)

X_train_enc = preprocessor.transform(X_train)
feature_names = get_feature_names(preprocessor)

print("Encoded train matrix:", X_train_enc.shape)
print("Feature names available:", len(feature_names))

# Show first few feature names (useful for SHAP later)
feature_names[:25]

c:\Users\Wassim Rahali\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: [1 7 8]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\Wassim Rahali\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: [11]. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
c:\Users\Wassim Rahali\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: [1 7 8]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\Wassim Rahali\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values

Encoded train matrix: (446917, 11289)
Feature names available: 11289


['num__Year',
 'num__Incident',
 'num__Victim Age',
 'num__Perpetrator Age',
 'num__Victim Count',
 'num__Perpetrator Count',
 'num__age_difference',
 'num__is_youth_victim',
 'num__same_race',
 'num__same_sex',
 'num__is_domestic',
 'num__multi_victim',
 'cat_ohe__Agency Code_AK00101',
 'cat_ohe__Agency Code_AK00102',
 'cat_ohe__Agency Code_AK00103',
 'cat_ohe__Agency Code_AK00104',
 'cat_ohe__Agency Code_AK00105',
 'cat_ohe__Agency Code_AK00106',
 'cat_ohe__Agency Code_AK00107',
 'cat_ohe__Agency Code_AK00108',
 'cat_ohe__Agency Code_AK00109',
 'cat_ohe__Agency Code_AK00110',
 'cat_ohe__Agency Code_AK00111',
 'cat_ohe__Agency Code_AK00113',
 'cat_ohe__Agency Code_AK00116']